# Fase 3 · Exploración y limpieza de Olist

Tablas a cargo de esta parte del proyecto: `order_items`, `products`, `sellers` y `category_translation`. Se lee `orders` únicamente para verificar la referencia desde los artículos. Ejecuta el notebook desde la raíz del proyecto con los CSV de [Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) en `datos/`.

In [2]:
from decimal import Decimal
from pathlib import Path

import pandas as pd
from IPython.display import display

data_dir = Path("datos")
input_files = {
    "order_items": "olist_order_items_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
primary_keys = {
    "order_items": ["order_id", "order_item_id"],
    "products": ["product_id"],
    "sellers": ["seller_id"],
    "category_translation": ["product_category_name"],
}
raw = {
    name: pd.read_csv(data_dir / filename, dtype="string", keep_default_na=False, na_values=[""])
    for name, filename in input_files.items()
}

## 1. Filas, columnas, huecos y duplicados

Cada fila de `order_items` representa un artículo dentro de un pedido. Su clave es el par `(order_id, order_item_id)`. Las otras tablas se identifican por el producto, el vendedor o el nombre portugués de la categoría. Se cargan los CSV como texto al comienzo para inspeccionar su formato antes de convertir los campos numéricos y de fecha.

In [3]:
overview = []
column_profiles = {}
for name, table in raw.items():
    blank_cells = table.apply(lambda column: column.str.fullmatch(r"\s*").sum())
    overview.append({
        "tabla": name,
        "filas": len(table),
        "columnas": len(table.columns),
        "celdas nulas": int(table.isna().sum().sum()),
        "solo espacios": int(blank_cells.sum()),
        "filas duplicadas": int(table.duplicated().sum()),
        "claves repetidas": int(table.duplicated(primary_keys[name]).sum()),
    })
    column_profiles[name] = pd.DataFrame({
        "nulos": table.isna().sum(),
        "solo espacios": blank_cells,
        "valores distintos": table.nunique(),
    })

display(pd.DataFrame(overview).set_index("tabla"))
display(pd.concat(column_profiles, names=["tabla", "columna"]))

,filas,columnas,celdas nulas,solo espacios,filas duplicadas,claves repetidas
tabla,,,,,,
order_items,112650,7,0,0,0,0
products,32951,9,2448,0,0,0
sellers,3095,4,0,0,0,0
category_translation,71,2,0,0,0,0


nulos  solo espacios  \
tabla                columna                                               
order_items          order_id                           0              0   
                     order_item_id                      0              0   
                     product_id                         0              0   
                     seller_id                          0              0   
                     shipping_limit_date                0              0   
                     price                              0              0   
                     freight_value                      0              0   
products             product_id                         0              0   
                     product_category_name            610              0   
                     product_name_lenght              610              0   
                     product_description_lenght       610              0   
                     product_photos_qty               610              0   
                     product_weight_g                   2              0   
                     product_length_cm                  2              0   
                     product_height_cm                  2              0   
                     product_width_cm                   2              0   
sellers              seller_id                          0              0   
                     seller_zip_code_prefix             0              0   
                     seller_city                        0              0   
                     seller_state                       0              0   
category_translation product_category_name              0              0   
                     product_category_name_english      0              0   

                                                    valores distintos  
tabla                columna                                           
order_items          order_id                                   98666  
                     order_item_id                                 21  
                     product_id                                 32951  
                     seller_id                                   3095  
                     shipping_limit_date                        93318  
                     price                                       5968  
                     freight_value                               6999  
products             product_id                                 32951  
                     product_category_name                         73  
                     product_name_lenght                           66  
                     product_description_lenght                  2960  
                     product_photos_qty                            19  
                     product_weight_g                            2204  
                     product_length_cm                             99  
                     product_height_cm                            102  
                     product_width_cm                              95  
sellers              seller_id                                   3095  
                     seller_zip_code_prefix                      2246  
                     seller_city                                  611  
                     seller_state                                  23  
category_translation product_category_name                         71  
                     product_category_name_english                 71

## 2. Claves, precios y campos de producto

Se quitan espacios en los extremos de los campos de texto y se eliminan las copias exactas. Si existen filas distintas con la misma clave, se detiene la ejecución para revisarlas. La edición descargada contiene cero filas duplicadas en las cuatro tablas.

Los valores monetarios se validan y se convierten a decimales exactos. El flete de cero es un valor válido. En productos se conservan las dimensiones y los pesos vacíos, y se contabilizan los pesos de cero. Se mantienen los encabezados originales para facilitar la carga posterior.

In [4]:
clean = {}
for name, table in raw.items():
    clean[name] = (
        table.apply(lambda column: column.str.strip().replace("", pd.NA))
        .drop_duplicates()
        .copy()
    )
    keys = clean[name][primary_keys[name]]
    if keys.isna().any().any() or keys.duplicated().any():
        raise ValueError(f"{name} contiene claves primarias nulas o repetidas.")

items = clean["order_items"]
items["order_item_id"] = items["order_item_id"].astype("Int64")
if items["order_item_id"].lt(1).any():
    raise ValueError("order_item_id debe ser positivo.")

items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"], format="%Y-%m-%d %H:%M:%S", errors="raise"
)
if items["shipping_limit_date"].isna().any():
    raise ValueError("Hay artículos sin fecha límite de despacho.")

money_summary = []
for column in ("price", "freight_value"):
    if not items[column].str.fullmatch(r"[0-9]+(?:\.[0-9]{1,2})?", na=False).all():
        raise ValueError(f"{column} contiene importes vacíos o con formato inválido.")
    items[column] = items[column].map(lambda value: Decimal(value).quantize(Decimal("0.01")))
    money_summary.append({
        "campo": column,
        "mínimo": items[column].min(),
        "máximo": items[column].max(),
        "ceros": int(items[column].eq(0).sum()),
    })
display(pd.DataFrame(money_summary).set_index("campo"))

products = clean["products"]
product_number_columns = [
    "product_name_lenght", "product_description_lenght", "product_photos_qty",
    "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
]
products[product_number_columns] = products[product_number_columns].astype("Int64")
display(pd.DataFrame({
    "nulos": products[product_number_columns].isna().sum(),
    "ceros": products[product_number_columns].eq(0).sum(),
}))

sellers = clean["sellers"]
if not sellers["seller_zip_code_prefix"].str.fullmatch(r"[0-9]{5}", na=False).all():
    raise ValueError("Se encontraron prefijos postales de vendedor distintos de cinco dígitos.")

,mínimo,máximo,ceros
campo,,,
price,0.85,6735.00,0
freight_value,0.00,409.68,383


,nulos,ceros
product_name_lenght,610,0
product_description_lenght,610,0
product_photos_qty,610,0
product_weight_g,2,4
product_length_cm,2,0
product_height_cm,2,0
product_width_cm,2,0


## 3. Categorías de producto en inglés

Se conservan como nulos los nombres de categoría que Olist dejó vacíos. Para los nombres en portugués que sí existen, se completa el catálogo con dos traducciones manuales. Las traducciones se guardan en `category_translation` para poder cruzarlas con `products` mediante `product_category_name`. `unknown` es una etiqueta de visualización para las consultas que agrupen esos productos.

In [5]:
category_key = "product_category_name"
english_name = "product_category_name_english"
categories = clean["category_translation"]
if categories[english_name].isna().any():
    raise ValueError("El catálogo contiene traducciones vacías.")

has_category = products[category_key].notna()
needs_translation = has_category & ~products[category_key].isin(categories[category_key])
missing = products.loc[needs_translation, category_key].value_counts()
manual_translations = {
    "pc_gamer": "gaming_pc",
    "portateis_cozinha_e_preparadores_de_alimentos":
        "portable_kitchen_appliances_and_food_preparation",
}
additions = missing.rename("productos").to_frame()
additions[english_name] = additions.index.map(manual_translations)
if additions[english_name].isna().any():
    raise ValueError("Quedan categorías con nombre pendientes de traducir.")

clean["category_translation"] = pd.concat(
    [categories, additions.reset_index()[[category_key, english_name]]],
    ignore_index=True,
)
display(additions)

product_categories = products[["product_id", category_key]].merge(
    clean["category_translation"],
    on=category_key,
    how="left",
    validate="many_to_one",
)
if product_categories.loc[product_categories[category_key].notna(), english_name].isna().any():
    raise ValueError("Hay categorías conocidas sin traducción tras el cruce.")
print(f"Productos sin categoría de origen: {(~has_category).sum():,}")
print(f"Productos con traducción añadida: {needs_translation.sum():,}")

,productos,product_category_name_english
product_category_name,,
portateis_cozinha_e_preparadores_de_alimentos,10,portable_kitchen_appliances_and_food_preparation
pc_gamer,3,gaming_pc


Productos sin categoría de origen: 610
Productos con traducción añadida: 13


## 4. Integridad y exportación

Se verifica que los artículos apunten a pedidos, productos y vendedores existentes, y que cada categoría informada en productos tenga una fila en el catálogo. Se escriben los resultados con los nombres originales en `datos/limpios/`. Los CSV de Kaggle permanecen en `datos/`.

In [6]:
order_ids = pd.read_csv(
    data_dir / "olist_orders_dataset.csv", usecols=["order_id"], dtype="string"
)["order_id"]
missing_references = {
    "order_items → orders": int((~items["order_id"].isin(order_ids)).sum()),
    "order_items → products": int((~items["product_id"].isin(products["product_id"])).sum()),
    "order_items → sellers": int((~items["seller_id"].isin(sellers["seller_id"])).sum()),
    "products → category_translation": int(
        (has_category & ~products[category_key].isin(clean["category_translation"][category_key])).sum()
    ),
}
display(pd.Series(missing_references, name="referencias sin coincidencia").to_frame())
if any(missing_references.values()):
    raise ValueError("Hay referencias sin coincidencia. Revisa las tablas antes de exportar.")

clean_dir = data_dir / "limpios"
clean_dir.mkdir(exist_ok=True)
exported = []
for name, filename in input_files.items():
    table = clean[name]
    target = clean_dir / filename
    table.to_csv(target, index=False, encoding="utf-8", date_format="%Y-%m-%d %H:%M:%S")
    exported.append({
        "tabla": name,
        "filas de origen": len(raw[name]),
        "filas exportadas": len(table),
        "archivo": target.as_posix(),
    })

display(pd.DataFrame(exported).set_index("tabla"))

,referencias sin coincidencia
order_items → orders,0
order_items → products,0
order_items → sellers,0
products → category_translation,0


,filas de origen,filas exportadas,archivo
tabla,,,
order_items,112650,112650,datos/limpios/olist_order_items_dataset.csv
products,32951,32951,datos/limpios/olist_products_dataset.csv
sellers,3095,3095,datos/limpios/olist_sellers_dataset.csv
category_translation,71,73,datos/limpios/product_category_name_translatio...
